# GLDAS database extraction on Google Earth Engine 
# GlobalVegetationCyclesProject

In [1]:
## ——Python API code

In [3]:
# import the libraries
import ee
# this is the package for visulization of the google earth engine maps in python
from ipyleaflet import *
from time import sleep
import multiprocessing
import math
import time
import sys
import calendar
import pandas as pd
import numpy as np
from pathlib import Path
from functools import partial
from contextlib import contextmanager
from ctypes import c_int
from multiprocessing import Value, Lock, Process

In [4]:
# intialize the google earth engine
ee.Initialize()

In [14]:
gldas_2_0 = ee.ImageCollection("NASA/GLDAS/V20/NOAH/G025/T3H").filterDate('1948-01-01','2000-01-01')
# load the GLDAS version 2.1 and filter the target years
gldas_2_1 = ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H").filterDate('1999-12-31','2025-01-01')
# Merge the two parts of the GLDAS data into one image collection
gldas = gldas_2_0.merge(gldas_2_1)


In [15]:
# check the date range of the merged GLDAS data
dateRange = gldas.reduceColumns(ee.Reducer.minMax(), ["system:time_start"])
# print(ee.Date(dateRange.get('max')).getInfo())

In [16]:
phenologySitesFull = ee.FeatureCollection("users/leonidmoore/GlobalVegetationCyclesProject/Selected_MODIS_pixels_20250314");
# show the head of the data
print(phenologySitesFull.limit(2).getInfo());

{'type': 'FeatureCollection', 'columns': {'ID': 'String', 'system:index': 'String', 'x': 'Float', 'y': 'Float'}, 'version': 1741981751913126, 'id': 'users/leonidmoore/GlobalVegetationCyclesProject/Selected_MODIS_pixels_20250314', 'properties': {'system:asset_size': 393865}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-80.87500118881532, 27.125001549679755]}, 'id': '0000000000000000188b', 'properties': {'ID': 'ID_6284', 'x': -80.875, 'y': 27.125}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-80.62500168825387, 27.12500154967975]}, 'id': '0000000000000000188c', 'properties': {'ID': 'ID_6285', 'x': -80.625, 'y': 27.125}}]}


In [25]:
# define the selected variables
selectVariables = ['Tair_f_inst',
                   'Rainf_f_tavg',
                   'Qair_f_inst',
                   'SoilMoi10_40cm_inst',
                   'Swnet_tavg']

### Below is the key part of the code

In [26]:
# here is the link where you can find the way to transform those different types data
# https://disc.gsfc.nasa.gov/information/faqs?title=How%20can%20I%20upscale%20the%203-hourly%20and%20hourly%20data%20from,%20respectively,%20GLDAS%20and%20NLDAS%20to%20calculate%20a%20daily%20value%3F

# For both the 3-hourly GLDAS data and the hourly NLDAS data, there are averaged, accumulated, and instantaneous variables. The following provides an example on how to calculate a daily value from these 3-hourly or hourly data, based on their day definition, which is 00Z-24Z.

# NB: The method described may need to be adjusted for uses of the data that require different day definitions.

# The averaged (or accumulated) variables for GLDAS data are backward-averaged (or accumulated), meaning each value represents the average (or accumulation) over the previous 3 hours. For example, the value given at 03Z represents the average (or accumulation) from 00Z-03Z. To calculate a daily value for an averaged (or accumulated) variable for GLDAS for a given day, average (or add) the values for 03Z, 06Z, 09Z, 12Z, 15Z, 18Z, and 21Z of that day and the value for 00Z of the following day (Fig. 1).

# To calculate a daily value for an instantaneous variable for GLDAS data, average all the values for a given day, i.e., 00z, 03z, 06z, 09z, 12z, 15z, 18z, and 21z (Fig. 2). Note that the method for calculating a daily value for instantaneous variables is slightly different from that for averaged or accumulated variables. 

# The above method for upscaling GLDAS data to calculate a daily value can also be applied to NLDAS data. The only difference is, instead of the eight 3-hourly values per day for GLDAS, there are 24 hourly values per day for NLDAS.

 

# FAQ1_2.jpg: Upscaling averaged and accumulated variables.

# Fig 1. Averaged and accumulated variables for GLDAS data are calculated every three hours, based on the data collected over the previous three hours.  To find a daily value for an averaged variable, calculate an average over all the 3-hourly data averages 03-21z for that day and 00z for the following day.  To find a daily value for an accumulated variable, add all the 3-hourly accumulation measurements from 03-21z for that day and 00z for the following day.

 

# FAQ2_2.jpg: Upscaling instantaneous variables.

# Fig 2. Instantaneous variables for GLDAS data are calculated every three hours, based on the data recorded at that 3-hourly time stamp.  To find a daily value for this variable, calculate an average over all the 3-hourly measurements 00-21z for that day.


In [27]:
for i in range(len(selectVariables)):
    # here words[i] is the array element
    # import the GLADIS data version 2 which ranges from 1948-2010;
    gldas_2_Chose = gldas.select(selectVariables[i])
    #print('Title 01:',gldas_2_Chose.first().getInfo())
    dataProjection = ee.Image(gldas.first()).projection()
    #print(dataProjection.getInfo())
    for year in range(2000,2025): # here is the place to change the year range#
        # Start and End Dates
        inidate = ee.Date.fromYMD(year,1,1)
        enddate = ee.Date.fromYMD((year+1),1,1)
        # Difference between start and end in days
        difdate = enddate.difference(inidate, 'day')
        # generate a list of the dates for everyday
        lapse = ee.List.sequence(0, difdate.subtract(1))
        # generate a list of days 
        def listDatesFunc(day):
            return inidate.advance(day,'day')
        listDates = lapse.map(listDatesFunc)
        # check the information of the listdates
        # print(listDates.getInfo())
   
        #us map function to do this data reduction from 3- hourly to daily
        #Only the temperature data needs the maximum and minimum
        # define the function
        def dailyDataDeriveMean (d):
            filtered = gldas_2_Chose.filterDate(ee.Date(d).getRange('day'))
            # here is the place you have to change to .mean(), .max() and min(),variance
            return filtered.mean().subtract(273).toFloat().reproject(crs=dataProjection).set('system:time_start', ee.Date(d))
        def dailyDataDeriveMin (d):
            filtered = gldas_2_Chose.filterDate(ee.Date(d).getRange('day'))
            # here is the place you have to change to .mean(), .max() and min(),variance
            return filtered.min().subtract(273).toFloat().reproject(crs=dataProjection).set('system:time_start', ee.Date(d))
        def dailyDataDeriveMax (d):
            filtered = gldas_2_Chose.filterDate(ee.Date(d).getRange('day'))
            # here is the place you have to change to .mean(), .max() and min(),variance
            return filtered.max().subtract(273).toFloat().reproject(crs=dataProjection).set('system:time_start', ee.Date(d))
        
        def dailyDataDeriveRain (d):
            dailyStart = ee.Date(d).advance(1,'hour')
            dailyEnd = ee.Date(d).advance(25,'hour')
            filtered = gldas_2_Chose.filterDate(dailyStart,dailyEnd)
            #here is the place you have to change to .mean(), .max() and min()
            return filtered.mean().multiply(86400).toFloat().reproject(crs=dataProjection).set('system:time_start',ee.Date(d))
        def dailyDataDeriveTavg (d):
            dailyStart = ee.Date(d).advance(1,'hour')
            dailyEnd = ee.Date(d).advance(25,'hour')
            filtered = gldas_2_Chose.filterDate(dailyStart,dailyEnd)
            #here is the place you have to change to .mean(), .max() and min()
            return filtered.mean().toFloat().reproject(crs=dataProjection).set('system:time_start',ee.Date(d))
        def dailyDataDeriveInt (d):
            filtered = gldas_2_Chose.filterDate(ee.Date(d).getRange('day'))
            # here is the place you have to change to .mean(), .max() and min()
            return filtered.mean().toFloat().reproject(crs=dataProjection).set('system:time_start',ee.Date(d))
        
        # subset the shapefile points with year property value, we don't need this filter by year anymore
        # then just use the full sites equals the phenology sites
        phenologySites = phenologySitesFull
        
        #temperature is different for it has the mean ,min and max
        if 'Tair_f_inst' == selectVariables[i]:
            # usw map function to do this data reduction from 3- hourly to daily
            # Only the temperature data needs the maximum and minimum
            # dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDerive))
            for vl in ['Mean','Min','Max']:
                if vl == 'Mean':
                    dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDeriveMean))
                elif vl == 'Min':
                    dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDeriveMin))
                else:
                    dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDeriveMax))
                # Transfer the imageCollection into a new multiband Image
                multibandImage = dailyComposites.toBands()
                if not year % 4:
                    dayNumber = 366
                else:
                    dayNumber = 365
                # generate the new band names by the day of the year
                newBandNames = [selectVariables[i]+'_'+vl+'_'+str(format(x, '03')) for x in range(1,dayNumber+1)]
                # rename the multibandImage with the new band names
                multibandImage = multibandImage.rename(newBandNames)
                #print(formalizedImage.bandNames().getInfo())
                # print(phenologySites.size().getInfo())
                # apply reduceRegions which is faster for points data
                sampledDailyVariable = multibandImage.reduceRegions(collection = phenologySites,
                                                                    reducer = ee.Reducer.first())
                # check the information of the output
                # print(sampledDailySoilMosture.first().getInfo())
                # export the out put to google cloud drive
                exportTask = ee.batch.Export.table.toCloudStorage(
                    collection = sampledDailyVariable,
                    description = "PhenologyData_Extraction_of_"+vl+"_"+str(year)+"_at_"+selectVariables[i],
                    fileNamePrefix = "GLDAS_for_Consti/Extracted_Daily_"+vl+"_Data_Year_"+str(year)+"_of_"+selectVariables[i],
                    bucket = "crowtherlab_gcsb_lidong",
                    fileFormat = 'csv')
                # start the export task
                exportTask.start()
                # show the task status
                exportTask.status()
        else :
            if 'Rainf_f_tavg' == selectVariables[i]: #//rain needs specical transformation
                #get the daily composite for rain with the function dailyDataDeriveRain 
                dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDeriveRain))
                # print(dailyComposites.first().getInfo())
            elif 'Rainf_f_tavg' != selectVariables[i] and 'tavg' in selectVariables[i]:
                #get the daily composite for '_tavg' but not for rain with the function dailyDataDeriveTavg 
                dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDeriveTavg))
            else :
                # #get the daily composite for '_inst' with the function dailyDataDeriveIn use the specific function 
                dailyComposites = ee.ImageCollection.fromImages(listDates.map(dailyDataDeriveInt))
            # Transfer the imageCollection into a new multiband Image
            multibandImage = dailyComposites.toBands()
            # print(multibandImage.bandNames().getInfo())
            # generate new band names list
            # get the day number in the specific year
            if not year % 4:
                dayNumber = 366
            else:
                dayNumber = 365
            # generate the new band names by the day of the year
            newBandNames = [selectVariables[i]+'_'+str(format(x, '03')) for x in range(1,dayNumber+1)]
            # rename the multibandImage with the new band names
            multibandImage = multibandImage.rename(newBandNames)
            #print(formalizedImage.bandNames().getInfo())

            # print(phenologySites.size().getInfo())
            # apply reduceRegions which is faster for points data
            sampledDailyVariable = multibandImage.reduceRegions(collection = phenologySites,
                                                                   reducer = ee.Reducer.first())
            # check the information of the output 
            # print(sampledDailySoilMosture.first().getInfo())
            # export the out put to google cloud drive
            exportTask = ee.batch.Export.table.toCloudStorage(
                collection = sampledDailyVariable,
                description = "PhenologyData_Extraction_of_"+str(year)+"_at_"+selectVariables[i],
                fileNamePrefix = "GLDAS_for_Consti/Extracted_Daily_Data_Year_"+str(year)+"_of_"+selectVariables[i],
                bucket = "crowtherlab_gcsb_lidong",
                fileFormat = 'csv')
            # start the export task
            exportTask.start()
            # show the task status
            exportTask.status()


## Bash downloading Google Cloud Storage data (On iMac)

In [ ]:
# this is the code for downloading data from Google cloud to loca drive in bash
for fileName in `gsutil ls gs://crowtherlab_gcsb_lidong/PhenologyTest`
do
#  echo gsutil cp gs://crowtherlab_gcsb_lidong/${fileName##*/} /Volumes/Scratch/Lidong_Mo/EcosystemPhotoRespProject/DerivedClimateData
 gsutil cp gs://crowtherlab_gcsb_lidong/PhenologyTest/${fileName##*/} /Volumes/Scratch/Lidong_Mo/AutaumPhenologyTier2/GLDAS_Extracted_Ver2 
done
                                                        
#                                                        